In [1]:
import sys
sys.path.append('../')
import time

import h5py
import epics
import numpy as np
import matplotlib.pyplot as plt

from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

Initialize cax control of siriuspy devices

In [2]:
CAX = CAXCtrl()

# Scan

Parameters

In [3]:
STEP = 0.0001 # [mm]
SCALE = 1     # [um/px]
DELAY = 5     # [s]
MAXERRORCOUNT = 5

lens_pv_sp = 'CAX:B:PP01:F.VAL'
lens_pv_rbv = 'CAX:B:PP01:F.RBV'

Functions

In [5]:
def dvfs_acquire():
    CAX.dvf_A1.cmd_acquire_on()
    CAX.dvf_B1.cmd_acquire_on()

def current_parameters():
    return {
        'slit1': {
            'top': CAX.slit_A1.top_pos,
            'bottom': CAX.slit_A1.bottom_pos,
            'left': CAX.slit_A1.left_pos,
            'right': CAX.slit_A1.right_pos
        },
        'dvf1': {
            'expo_time': CAX.dvf_A1.acquisition_time
        },
        'dvf2': {
            'expo_time': CAX.dvf_A1.acquisition_time,
            'z_pos': CAX.dvf_B1.z_pos,
            # 'lens_pos': 
        },
        'mirror': {
            'ry': CAX.mirror.ry_pos,
            'tx': CAX.mirror.tx_pos,
            'y1': CAX.mirror.y1_pos,
            'y2': CAX.mirror.y2_pos,
            'y3': CAX.mirror.y3_pos,
            'photocollector': CAX.mirror.photocurrent_signal
        }
    }

def fwhm(data):
    threshold = 0.5*np.max(data)
    mask = data > threshold
    return np.sum(mask) * SCALE

def peak(data):
    return np.max(data)

def position(data):
    return np.argmax(data) * SCALE

def move_all_slits(top, bottom, left, right):
    CAX.slit_A1.move_top(value=top)
    CAX.slit_A1.move_bottom(value=bottom)
    CAX.slit_A1.move_left(value=left)
    CAX.slit_A1.move_right(value=right)

def open_all_slits():
    move_all_slits(top=19.7-0.04,
                   bottom=35.8,
                   left=43.6-0.04,
                   right=47.2)

def move(ry_pos,delay=DELAY):
    CAX.mirror.ry_pos = ry_pos
    time.sleep(delay)

def move_step(step=STEP,delay=DELAY):
    pos = CAX.mirror.ry_pos + step
    move(ry_pos=pos,delay=delay)


def get_image(dvf):

    count = 0
    while count < MAXERRORCOUNT:
        try:
            if not dvf.acquisition_status:
                dvf.cmd_acquire_on()
            return dvf.image
        except Exception as err:
            print(f" WARNING. When trying to fetch image from DVF1: {err} ")
            time.sleep(2)
            count += 1
            if count < MAXERRORCOUNT:
                print("\n Repeating the procedure...\n")
            else:
                raise Exception("Client exception")



Initial state

In [6]:
local_time = time.localtime()
formatted_time = time.strftime("%Y%m%d-%H%M%S", local_time)
formatted_date = time.strftime("%Y%m%d", local_time)

formatted_time, formatted_date

('20250924-160258', '20250924')

In [7]:
ANALYSIS = 'lens'

In [ ]:
parameters0 = current_parameters()

states_dir = '../states/'
state_name = f'{formatted_time}-{ANALYSIS}.txt'

with open(states_dir+state_name, 'w') as f:
    f.write(str(parameters0))

print(parameters0)

Initialize scan file

In [8]:
scaname = f'scan_lens_{formatted_date}.h5'
datadir  = '/home/ids/data/'
direc    = f'{formatted_date}-Lens/'

In [46]:
file = HDF5File(filename=scaname,filedir=datadir+direc)

file.save_metadata(metadata_dict={
    'lens_pos': epics.caget(lens_pv_rbv)
})

Execution

In [47]:
positions = np.linspace(3.6,4.4,100)

In [48]:
t0 = time.time()


for i, pos in enumerate(positions):
    print(f'{i}/{len(positions)-1}')

    #!
    #todo: deslocar feixe ate a borda. como está agora é
    #todo: a partir do valor inicial, mas deve ser a partir da borda

    epics.caput(pvname=lens_pv_sp, value=pos)
    
    time.sleep(2)

    img2 = get_image(dvf=CAX.dvf_B1)
    expotime2 = CAX.dvf_B1.exposure_time
    
    info = {
        'lens_pos': epics.caget(lens_pv_rbv),
        'expo_time': expotime2
    }
    
    file.save_dataset(dsetname=f'lens-{i:04d}', dsetmetadata=info, dsetdata=img2)


t1 = time.time()

print()
print(f'elapsed time [min]: {(t1-t0)/60}')

0/99
1/99
2/99
3/99
4/99
5/99
6/99
7/99
8/99
9/99
10/99
11/99
12/99
13/99
14/99
15/99
16/99
17/99
18/99
19/99
20/99
21/99
22/99
23/99
24/99
25/99
26/99
27/99
28/99
29/99
30/99
31/99
32/99
33/99
34/99
35/99
36/99
37/99
38/99
39/99
40/99
41/99
42/99
43/99
44/99
45/99
46/99
47/99
48/99
49/99
50/99
51/99
52/99
53/99
54/99
55/99
56/99
57/99
58/99
59/99
60/99
61/99
62/99
63/99
64/99
65/99
66/99
67/99
68/99
69/99
70/99
71/99
72/99
73/99
74/99
75/99
76/99
77/99
78/99
79/99
80/99
81/99
82/99
83/99
84/99
85/99
86/99
87/99
88/99
89/99
90/99
91/99
92/99
93/99
94/99
95/99
96/99
97/99
98/99
99/99

elapsed time [min]: 4.870182573795319


In [49]:
filename = '/'.join([datadir+direc,scaname])

f = h5py.File(name=filename)

In [50]:
f.keys()

<KeysViewHDF5 ['lens-0000', 'lens-0001', 'lens-0002', 'lens-0003', 'lens-0004', 'lens-0005', 'lens-0006', 'lens-0007', 'lens-0008', 'lens-0009', 'lens-0010', 'lens-0011', 'lens-0012', 'lens-0013', 'lens-0014', 'lens-0015', 'lens-0016', 'lens-0017', 'lens-0018', 'lens-0019', 'lens-0020', 'lens-0021', 'lens-0022', 'lens-0023', 'lens-0024', 'lens-0025', 'lens-0026', 'lens-0027', 'lens-0028', 'lens-0029', 'lens-0030', 'lens-0031', 'lens-0032', 'lens-0033', 'lens-0034', 'lens-0035', 'lens-0036', 'lens-0037', 'lens-0038', 'lens-0039', 'lens-0040', 'lens-0041', 'lens-0042', 'lens-0043', 'lens-0044', 'lens-0045', 'lens-0046', 'lens-0047', 'lens-0048', 'lens-0049', 'lens-0050', 'lens-0051', 'lens-0052', 'lens-0053', 'lens-0054', 'lens-0055', 'lens-0056', 'lens-0057', 'lens-0058', 'lens-0059', 'lens-0060', 'lens-0061', 'lens-0062', 'lens-0063', 'lens-0064', 'lens-0065', 'lens-0066', 'lens-0067', 'lens-0068', 'lens-0069', 'lens-0070', 'lens-0071', 'lens-0072', 'lens-0073', 'lens-0074', 'lens-0075

In [55]:
dict(f['lens-0000'].attrs)

{'expo_time': 0.05, 'lens_pos': 3.6000086700000002}

In [56]:
fwhmsx = []
fwhmsy = []

for lens in f:
    img = np.array(f[lens])
    projx = np.sum(img,axis=0)
    projy = np.sum(img,axis=1)
    
    fwhmsx.append(fwhm(projx))
    fwhmsy.append(fwhm(projy))

lens_poses = [f[lens].attrs['lens_pos'] for lens in f]

In [59]:
%matplotlib qt

In [60]:
plt.plot(lens_poses,fwhmsx)
plt.plot(lens_poses,fwhmsy)
# plt.xlim()
plt.grid()
plt.show()

In [65]:
import numpy as np

prmx = np.polyfit(lens_poses, fwhmsx, 2)
prmy = np.polyfit(lens_poses, fwhmsy, 2)

In [67]:
xv, yv = -prmx[1]/(2*prmx[0]), -prmy[1]/(2*prmy[0])

In [68]:
(xv + yv) / 2

3.954854961144046

In [69]:
epics.caput(lens_pv_sp, 3.95485496114)

1